## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:王潇


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

namespace {
constexpr int LIM = 1024 + 5;

struct FastScanner {
    static int nextInt() {
        int c = getchar();
        while (c < '0' || c > '9') c = getchar();
        int x = 0;
        while (c >= '0' && c <= '9') {
            x = x * 10 + (c - '0');
            c = getchar();
        }
        return x;
    }
};

struct RecursivePlan {
    int n = 0;
    int a[LIM]{};
    vector<int> ops;

    static void compressXorOps(vector<int>& seq) {
        vector<int> merged;
        merged.reserve(seq.size());
        for (int x : seq) {
            if (!merged.empty() && x < 0 && merged.back() < 0) {
                merged.back() = -((-merged.back()) ^ (-x));
                if (merged.back() == 0) merged.pop_back();
            } else {
                merged.push_back(x);
            }
        }
        seq.swap(merged);
    }

    bool solve() {
        bool vis[100005] = {};
        for (int i = 0; i < n; ++i) vis[i] = true;
        for (int i = 0; i < n; ++i) {
            if (!vis[i]) return false;
        }

        if (n == 1) return true;

        RecursivePlan left, right;
        left.n = right.n = n >> 1;

        for (int i = 0; i < (n >> 1); ++i) {
            left.a[i] = a[i << 1] >> 1;
            right.a[i] = a[i << 1 | 1] >> 1;
        }

        if (!left.solve() || !right.solve()) return false;

        if (a[0] & 1) {
            ops.push_back(n == 2 ? 1 : -1);
        }

        int xorL = 0;
        for (int x : left.ops) {
            if (x > 0) {
                ops.push_back(-1);
                ops.push_back(1);
            } else {
                ops.push_back(x * 2);
                xorL ^= (-x) * 2;
            }
        }
        if (xorL) ops.push_back(-xorL);

        int xorR = 0;
        for (int x : right.ops) {
            if (x > 0) {
                ops.push_back(1);
                ops.push_back(-1);
            } else {
                ops.push_back(x * 2);
                xorR ^= (-x) * 2;
            }
        }

        if ((xorR & (n >> 1)) != (xorL & (n >> 1))) return false;
        if (xorL >= (n >> 1)) xorL -= (n >> 1);
        if (xorR >= (n >> 1)) xorR -= (n >> 1);
        if (xorL != xorR) return false;

        compressXorOps(ops);
        return true;
    }
};

class Solver {
public:
    void run() {
        readInput();
        refreshPos();

        blockLen = (A - B + n) % n;
        blockLen &= -blockLen;
        if (blockLen == 0) blockLen = n;

        if (!solveLowBits()) {
            puts("-1");
            return;
        }
        if (!sortByBuckets()) {
            puts("-1");
            return;
        }

        for (int i = 0; i < n; ++i) {
            assert(p[i] == i);
        }

        printAnswer();
    }

private:
    int n{}, A{}, B{}, blockLen{};
    int p[LIM]{}, pos[LIM]{};
    vector<int> ans;

    void readInput() {
        n = FastScanner::nextInt();
        A = FastScanner::nextInt();
        B = FastScanner::nextInt();
        for (int i = 0; i < n; ++i) {
            p[i] = FastScanner::nextInt();
        }
    }

    void refreshPos() {
        for (int i = 0; i < n; ++i) {
            pos[p[i]] = i;
        }
    }

    void applySwapMagic() {
        ans.push_back(0);
        for (int i = 0; i < n; ++i) {
            if (p[i] == A) p[i] = B;
            else if (p[i] == B) p[i] = A;
        }
        refreshPos();
    }

    void applyAddMagic(int v) {
        v %= n;
        if (v < 0) v += n;
        if (v == 0) return;

        ans.push_back(v);
        for (int i = 0; i < n; ++i) {
            p[i] += v;
            if (p[i] >= n) p[i] -= n;
        }
        refreshPos();
    }

    void applyXorMagic(int v) {
        if (v == 0) return;

        ans.push_back(-v);
        for (int i = 0; i < n; ++i) {
            p[i] ^= v;
        }
        refreshPos();
    }

    bool solveLowBits() {
        if (blockLen <= 1) return true;

        RecursivePlan plan;
        plan.n = blockLen;
        for (int i = 0; i < n; ++i) {
            plan.a[i] = p[i] & (blockLen - 1);
        }

        if (!plan.solve()) return false;

        for (int op : plan.ops) {
            if (op > 0) applyAddMagic(op);
            else applyXorMagic(-op);
        }
        return true;
    }

    pair<int, int> locatePair(int x, int y) const {
        int delta = (y - x + n - blockLen + n) % n;
        int px = 0, py = 0;

        for (int step = n >> 1; step >= 2 * blockLen; step >>= 1) {
            if (delta >= step) {
                delta -= step;
                py += step >> 1;
            } else {
                px += step >> 1;
            }
        }

        int low = x & (blockLen - 1);
        px += (n >> 1) + low;
        py += low;
        return {px, py};
    }

    void doSwap(int x, int y) {
        if (((x / blockLen) & 1) == ((y / blockLen) & 1)) {
            int mid = (((x / blockLen) & 1) == 0)
                        ? ((x & (blockLen - 1)) + blockLen)
                        : (x & (blockLen - 1));
            doSwap(x, mid);
            doSwap(y, mid);
            doSwap(x, mid);
            return;
        }

        auto [pA, _1] = locatePair(A, B);
        auto [pX, _2] = locatePair(x, y);
        (void)_1;
        (void)_2;

        applyAddMagic((pX - x + n) % n);
        applyXorMagic(pX ^ pA);
        applyAddMagic((A - pA + n) % n);

        applySwapMagic();

        applyAddMagic((pA - A + n) % n);
        applyXorMagic(pX ^ pA);
        applyAddMagic((x - pX + n) % n);
    }

    bool checkBucket(int rem) const {
        vector<int> bucket;
        for (int j = rem; j < n; j += blockLen) {
            bucket.push_back(p[j]);
        }
        sort(bucket.begin(), bucket.end());

        int idx = 0;
        for (int j = rem; j < n; j += blockLen) {
            if (bucket[idx++] != j) return false;
        }
        return true;
    }

    bool sortByBuckets() {
        for (int rem = 0; rem < blockLen; ++rem) {
            if (!checkBucket(rem)) return false;

            for (int j = rem; j < n; j += blockLen) {
                if (p[j] != j) doSwap(j, p[j]);
            }
        }
        return true;
    }

    void printAnswer() const {
        printf("%d\n", (int)ans.size());
        for (int x : ans) {
            if (x == 0) {
                printf("0\n");
            } else if (x < 0) {
                printf("1 %d\n", -x);
            } else {
                printf("2 %d\n", x);
            }
        }
    }
};
}

int main() {
    Solver solver;
    solver.run();
    return 0;
}

## B 长跑

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int N, L, Maxn, S;

    while (cin >> N >> L >> Maxn >> S) {
        vector<pair<int, int>> stations;
        stations.reserve(N);

        for (int i = 0; i < N; ++i) {
            int p, c;
            cin >> p >> c;
            stations.push_back({p, c});
        }

        if (L == 0) {
            cout << "Yes\n";
            continue;
        }

        if (Maxn == 0) {
            cout << "No\n";
            continue;
        }

        sort(stations.begin(), stations.end());
        vector<int> pos, cost;
        for (int i = 0; i < N; ) {
            int p = stations[i].first;
            int mn = stations[i].second;
            int j = i + 1;
            while (j < N && stations[j].first == p) {
                mn = min(mn, stations[j].second);
                ++j;
            }
            pos.push_back(p);
            cost.push_back(mn);
            i = j;
        }

        int M = (int)pos.size();
        const int INF = 1e9;
        vector<int> dp(M, INF);

        for (int i = 0; i < M; ++i) {
            if (pos[i] <= Maxn) {
                dp[i] = cost[i];
            }

            for (int j = 0; j < i; ++j) {
                if (pos[i] - pos[j] <= Maxn && dp[j] != INF) {
                    dp[i] = min(dp[i], dp[j] + cost[i]);
                }
            }
        }

        bool ok = false;

        if (L <= Maxn && 0 <= S) {
            ok = true;
        }

        for (int i = 0; i < M && !ok; ++i) {
            if (L - pos[i] <= Maxn && dp[i] <= S) {
                ok = true;
            }
        }

        cout << (ok ? "Yes" : "No") << '\n';
    }

    return 0;
}

## C 最长回文

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

using ll = long long;

const ll MOD1 = 1000000007;
const ll MOD2 = 1000000009;
const ll BASE = 911382323;

struct Hash {
    vector<ll> h1, h2;

    Hash() {}

    Hash(const string& s, const vector<ll>& p1, const vector<ll>& p2) {
        int n = s.size();
        h1.assign(n + 1, 0);
        h2.assign(n + 1, 0);

        for (int i = 0; i < n; i++) {
            int v = s[i] + 1;
            h1[i + 1] = (h1[i] * BASE + v) % MOD1;
            h2[i + 1] = (h2[i] * BASE + v) % MOD2;
        }
    }

    pair<ll, ll> get(int l, int len, const vector<ll>& p1, const vector<ll>& p2) const {
        if (len <= 0) return {0, 0};

        ll x1 = (h1[l + len] - h1[l] * p1[len]) % MOD1;
        if (x1 < 0) x1 += MOD1;

        ll x2 = (h2[l + len] - h2[l] * p2[len]) % MOD2;
        if (x2 < 0) x2 += MOD2;

        return {x1, x2};
    }
};

void manacher(const string& s, vector<int>& odd, vector<int>& even) {
    int n = s.size();

    odd.assign(n, 0);
    even.assign(n, 0);

    int l = 0, r = -1;

    for (int i = 0; i < n; i++) {
        int k = 1;

        if (i <= r) {
            k = min(odd[l + r - i], r - i + 1);
        }

        while (i - k >= 0 && i + k < n && s[i - k] == s[i + k]) {
            k++;
        }

        odd[i] = k;

        if (i + k - 1 > r) {
            l = i - k + 1;
            r = i + k - 1;
        }
    }

    l = 0;
    r = -1;

    for (int i = 0; i < n; i++) {
        int k = 0;

        if (i <= r) {
            k = min(even[l + r - i + 1], r - i + 1);
        }

        while (i - k - 1 >= 0 && i + k < n && s[i - k - 1] == s[i + k]) {
            k++;
        }

        even[i] = k;

        if (i + k - 1 > r) {
            l = i - k;
            r = i + k - 1;
        }
    }
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    string A, B;

    cin >> n;
    cin >> A;
    cin >> B;

    vector<ll> pow1(n + 1), pow2(n + 1);
    pow1[0] = 1;
    pow2[0] = 1;

    for (int i = 1; i <= n; i++) {
        pow1[i] = pow1[i - 1] * BASE % MOD1;
        pow2[i] = pow2[i - 1] * BASE % MOD2;
    }

    string revA = A;
    reverse(revA.begin(), revA.end());

    Hash hashRevA(revA, pow1, pow2);
    Hash hashB(B, pow1, pow2);

    auto lcp = [&](int aEnd, int bStart) -> int {
        if (aEnd < 0 || bStart >= n) return 0;

        int posRevA = n - 1 - aEnd;
        int limit = min(aEnd + 1, n - bStart);

        int left = 0;
        int right = limit;

        while (left < right) {
            int mid = (left + right + 1) / 2;

            auto hA = hashRevA.get(posRevA, mid, pow1, pow2);
            auto hB = hashB.get(bStart, mid, pow1, pow2);

            if (hA == hB) {
                left = mid;
            } else {
                right = mid - 1;
            }
        }

        return left;
    };

    vector<int> oddA, evenA, oddB, evenB;
    manacher(A, oddA, evenA);
    manacher(B, oddB, evenB);

    int ans = 1;

    for (int k = 0; k < n; k++) {
        ans = max(ans, 2 * lcp(k, k));
    }

    for (int i = 0; i < n; i++) {
        int r = oddA[i];

        int L = i - r + 1;
        int R = i + r - 1;
        int len = 2 * r - 1;

        ans = max(ans, len + 2 * lcp(L - 1, R));
    }

    for (int i = 0; i < n; i++) {
        int r = evenA[i];

        if (r == 0) continue;

        int L = i - r;
        int R = i + r - 1;
        int len = 2 * r;

        ans = max(ans, len + 2 * lcp(L - 1, R));
    }

    for (int i = 0; i < n; i++) {
        int r = oddB[i];

        int L = i - r + 1;
        int R = i + r - 1;
        int len = 2 * r - 1;

        ans = max(ans, len + 2 * lcp(L, R + 1));
    }

    for (int i = 0; i < n; i++) {
        int r = evenB[i];

        if (r == 0) continue;

        int L = i - r;
        int R = i + r - 1;
        int len = 2 * r;

        ans = max(ans, len + 2 * lcp(L, R + 1));
    }

    cout << ans << '\n';

    return 0;
}

## D 优惠券

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

const int MAX_ID = 100000 + 5;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int m;

    while (cin >> m) {
        vector<int> lastPos(MAX_ID, 0);
        vector<string> lastOp(MAX_ID);

        set<int> unknown;

        int ans = -1;
        bool valid = true;

        auto takeUnknownAfter = [&](int pos) -> bool {
            auto it = unknown.upper_bound(pos);
            if (it == unknown.end()) {
                return false;
            }
            unknown.erase(it);
            return true;
        };

        for (int i = 1; i <= m; i++) {
            string op;
            cin >> op;

            if (op == "?" || op == "？") {
                if (valid) {
                    unknown.insert(i);
                }
                continue;
            }

            int x;
            cin >> x;

            if (!valid) {
                continue;
            }

            if (op == "o") {
                op = "O";
            }

            if (lastPos[x] == 0) {
                if (op == "O") {
                    if (!takeUnknownAfter(0)) {
                        ans = i;
                        valid = false;
                    }
                }
            } else {
                if (lastOp[x] == op) {
                    if (!takeUnknownAfter(lastPos[x])) {
                        ans = i;
                        valid = false;
                    }
                }
            }

            lastPos[x] = i;
            lastOp[x] = op;
        }

        cout << ans << '\n';
    }

    return 0;
}

## E 任意点

In [ ]:
## add your code here
#include <iostream>
#include <vector>
using namespace std;

const int MAXN = 105;
int parent[MAXN];     

int find(int x) {
    if (parent[x] != x) {
        parent[x] = find(parent[x]);
    }
    return parent[x];
}

void unite(int x, int y) {
    x = find(x);
    y = find(y);
    if (x != y) {
        parent[y] = x;
    }
}

int main() {
    int n;
    cin >> n;
    vector<pair<int, int>> points(n);
    
    for (int i = 0; i < n; ++i) {
        parent[i] = i;
    }
    
    for (int i = 0; i < n; ++i) {
        cin >> points[i].first >> points[i].second;
    }
    
    for (int i = 0; i < n; ++i) {
        for (int j = i + 1; j < n; ++j) {
            if (points[i].first == points[j].first || points[i].second == points[j].second) {
                unite(i, j);
            }
        }
    }
    
    int cnt = 0;
    for (int i = 0; i < n; ++i) {
        if (find(i) == i) {
            cnt++;
        }
    }
    
    cout << cnt - 1 << endl;
    
    return 0;
}

## F 通配符匹配

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

struct Piece {
    string s;
    vector<int> pi;
};

struct Part {
    int offset;
    int pieceId;
};

struct Segment {
    int len;
    vector<Part> parts;
};

vector<Piece> pieces;
vector<Segment> segments;
bool leadingStar = false;
bool trailingStar = false;
bool hasStar = false;

int getPieceId(const string& t) {
    for (int i = 0; i < (int)pieces.size(); i++) {
        if (pieces[i].s == t) return i;
    }

    Piece p;
    p.s = t;
    p.pi.assign(t.size(), 0);

    for (int i = 1; i < (int)t.size(); i++) {
        int j = p.pi[i - 1];
        while (j > 0 && t[i] != t[j]) j = p.pi[j - 1];
        if (t[i] == t[j]) j++;
        p.pi[i] = j;
    }

    pieces.push_back(p);
    return (int)pieces.size() - 1;
}

Segment buildSegment(const string& t) {
    Segment seg;
    seg.len = (int)t.size();

    int i = 0;
    while (i < (int)t.size()) {
        if (t[i] == '?') {
            i++;
            continue;
        }

        int st = i;
        while (i < (int)t.size() && t[i] != '?') {
            i++;
        }

        string block = t.substr(st, i - st);
        int id = getPieceId(block);
        seg.parts.push_back({st, id});
    }

    return seg;
}

void parsePattern(const string& pattern) {
    int m = pattern.size();

    hasStar = pattern.find('*') != string::npos;
    leadingStar = (m > 0 && pattern[0] == '*');
    trailingStar = (m > 0 && pattern[m - 1] == '*');

    string cur;

    for (char c : pattern) {
        if (c == '*') {
            if (!cur.empty()) {
                segments.push_back(buildSegment(cur));
                cur.clear();
            }
        } else {
            cur.push_back(c);
        }
    }

    if (!cur.empty()) {
        segments.push_back(buildSegment(cur));
    }
}

vector<vector<unsigned char>> buildOccurrences(const string& text) {
    int n = text.size();
    vector<vector<unsigned char>> occ(pieces.size());

    for (int id = 0; id < (int)pieces.size(); id++) {
        const string& p = pieces[id].s;
        int m = p.size();

        if (m > n) {
            continue;
        }

        occ[id].assign(n - m + 1, 0);

        int j = 0;
        for (int i = 0; i < n; i++) {
            while (j > 0 && text[i] != p[j]) {
                j = pieces[id].pi[j - 1];
            }

            if (text[i] == p[j]) {
                j++;
            }

            if (j == m) {
                occ[id][i - m + 1] = 1;
                j = pieces[id].pi[j - 1];
            }
        }
    }

    return occ;
}

bool matchSegmentAt(
    const Segment& seg,
    int pos,
    int textLen,
    const vector<vector<unsigned char>>& occ
) {
    if (pos < 0 || pos + seg.len > textLen) {
        return false;
    }

    for (const Part& part : seg.parts) {
        int start = pos + part.offset;
        int id = part.pieceId;

        if (start < 0 || start >= (int)occ[id].size()) {
            return false;
        }

        if (!occ[id][start]) {
            return false;
        }
    }

    return true;
}

int findEarliest(
    const Segment& seg,
    int startPos,
    int endLimit,
    int textLen,
    const vector<vector<unsigned char>>& occ
) {
    int maxStart = endLimit - seg.len;

    for (int pos = startPos; pos <= maxStart; pos++) {
        if (matchSegmentAt(seg, pos, textLen, occ)) {
            return pos;
        }
    }

    return -1;
}

bool matchFile(const string& text) {
    int n = text.size();

    auto occ = buildOccurrences(text);

    if (!hasStar) {
        if (segments.empty()) {
            return n == 0;
        }

        if (segments[0].len != n) {
            return false;
        }

        return matchSegmentAt(segments[0], 0, n, occ);
    }

    if (segments.empty()) {
        return true;
    }

    int m = segments.size();

    int cur = 0;
    int left = 0;
    int right = m - 1;
    int endLimit = n;

    if (!leadingStar) {
        if (!matchSegmentAt(segments[0], 0, n, occ)) {
            return false;
        }

        cur = segments[0].len;
        left = 1;
    }

    if (!trailingStar) {
        const Segment& lastSeg = segments[m - 1];
        int start = n - lastSeg.len;

        if (!matchSegmentAt(lastSeg, start, n, occ)) {
            return false;
        }

        endLimit = start;
        right = m - 2;
    }

    if (cur > endLimit) {
        return false;
    }

    for (int i = left; i <= right; i++) {
        int pos = findEarliest(segments[i], cur, endLimit, n, occ);
        if (pos == -1) {
            return false;
        }

        cur = pos + segments[i].len;
    }

    return cur <= endLimit;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string pattern;
    cin >> pattern;

    parsePattern(pattern);

    int n;
    cin >> n;

    while (n--) {
        string filename;
        cin >> filename;

        if (matchFile(filename)) {
            cout << "YES\n";
        } else {
            cout << "NO\n";
        }
    }

    return 0;
}

## G 汉诺塔

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

using i128 = __int128_t;

int rk[3][3];
int toPeg[35][3];
i128 dp[35][3];

int id(char c) {
    return c - 'A';
}

int thirdPeg(int a, int b) {
    for (int i = 0; i < 3; i++) {
        if (i != a && i != b) return i;
    }
    return -1;
}

void printInt128(i128 x) {
    if (x == 0) {
        cout << 0;
        return;
    }

    string s;
    while (x > 0) {
        s.push_back(char('0' + x % 10));
        x /= 10;
    }

    reverse(s.begin(), s.end());
    cout << s;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;

    for (int i = 0; i < 3; i++) {
        for (int j = 0; j < 3; j++) {
            rk[i][j] = 100;
        }
    }

    for (int i = 0; i < 6; i++) {
        string op;
        cin >> op;

        int a = id(op[0]);
        int b = id(op[1]);

        rk[a][b] = i;
    }

    for (int s = 0; s < 3; s++) {
        int best = -1;

        for (int t = 0; t < 3; t++) {
            if (s == t) continue;

            if (best == -1 || rk[s][t] < rk[s][best]) {
                best = t;
            }
        }

        toPeg[1][s] = best;
        dp[1][s] = 1;
    }

    for (int k = 2; k <= n; k++) {
        for (int s = 0; s < 3; s++) {
            int small = toPeg[k - 1][s];
            i128 cost = dp[k - 1][s];

            int big = thirdPeg(s, small);
            cost++;

            bool vis[3][3] = {};

            while (true) {
                if (vis[small][big]) {
                    break;
                }

                vis[small][big] = true;

                int nxt = toPeg[k - 1][small];
                cost += dp[k - 1][small];

                if (nxt == big) {
                    toPeg[k][s] = big;
                    dp[k][s] = cost;
                    break;
                }

                cost++;

                int oldSmall = small;
                small = nxt;
                big = oldSmall;
            }
        }
    }

    printInt128(dp[n][0]);
    cout << '\n';

    return 0;
}

## H 马步距离

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

using ll = long long;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    ll xp, yp, xs, ys;
    cin >> xp >> yp >> xs >> ys;

    ll dx = llabs(xp - xs);
    ll dy = llabs(yp - ys);

    if (dx < dy) swap(dx, dy);

    if (dx == 1 && dy == 0) {
        cout << 3 << '\n';
        return 0;
    }

    if (dx == 2 && dy == 2) {
        cout << 4 << '\n';
        return 0;
    }

    ll ans = max((dx + 1) / 2, (dx + dy + 2) / 3);

    while ((ans + dx + dy) % 2 != 0) {
        ans++;
    }

    cout << ans << '\n';

    return 0;
}

## I 直方图最大矩形

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

class Solution {
public:
    int largestRectangleArea(vector<int>& heights) {
        int n = heights.size();
        if (n == 0) return 0;

        stack<int> st;
        int ans = 0;

        heights.push_back(0);

        for (int i = 0; i < (int)heights.size(); i++) {
            while (!st.empty() && heights[i] < heights[st.top()]) {
                int h = heights[st.top()];
                st.pop();

                int left = st.empty() ? -1 : st.top();
                int width = i - left - 1;

                ans = max(ans, h * width);
            }

            st.push(i);
        }

        heights.pop_back();
        return ans;
    }
};

## J 消防局的设立

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    
    int n;
    if (!(cin >> n)) return 0;
    
    vector<vector<int>> adj(n + 1);
    for (int i = 2; i <= n; ++i) {
        int p; 
        cin >> p;
        adj[i].push_back(p);
        adj[p].push_back(i);
    }
    
    vector<int> depth(n + 1, -1), parent(n + 1, 0);
    queue<int> q;
    q.push(1);
    depth[1] = 0;
    while (!q.empty()) {
        int u = q.front(); q.pop();
        for (int v : adj[u]) {
            if (depth[v] == -1) {
                depth[v] = depth[u] + 1;
                parent[v] = u;
                q.push(v);
            }
        }
    }
    
    vector<int> order(n);
    iota(order.begin(), order.end(), 1);
    sort(order.begin(), order.end(), [&](int a, int b) {
        return depth[a] > depth[b];
    });
    
    vector<char> covered(n + 1, 0);
    int ans = 0;
    
    for (int u : order) {
        if (covered[u]) continue;
        
        int g;
        if (depth[u] >= 2)       g = parent[parent[u]]; 
        else if (depth[u] == 1)  g = parent[u];        
        else                     g = u;               
        
        ++ans;
        covered[g] = 1;
        for (int v : adj[g]) {
            covered[v] = 1;
            for (int w : adj[v]) {
                covered[w] = 1;
            }
        }
    }
    
    cout << ans << "\n";
    return 0;
}